# Validação GPU — JAX Unified JIT (v1.5.0 → v1.5.1)

**Projeto:** Geosteering AI v2.0 — Simulador Python  
**Sprint:** 10 Phase 2 — Gate de validação GPU para flip do default  
**Versão:** v1.5.0 (PR #24 mergeado) → candidato v1.5.1  
**Autor:** Daniel Leal  

## Objetivo

Validar 3 gates obrigatórios antes de flip `jax_strategy` default `"bucketed"` → `"unified"` em v1.5.1:

| Gate | Métrica | Target | Status |
|:-----|:--------|:------:|:------:|
| **G1** | XLA programs oklahoma_28 | == 1 (unified) | ⏳ |
| **G2** | VRAM peak unified oklahoma_28 | < 1 GB | ⏳ |
| **G3** | Throughput unified vs bucketed GPU | ≥ 3× | ⏳ |
| **G4** | Paridade numérica `\|H_b - H_u\|` | < 1e-10 | ⏳ |
| **G5** | jacfwd unified oklahoma_28 alta-ρ | 0 NaN/Inf | ⏳ |

## Instruções

1. `Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4` (ou A100 se disponível)
2. Execute todas as células em sequência (`Ctrl+F9` ou `Ambiente de execução → Executar tudo`)
3. Ao final, leia o **Relatório GO/NO-GO** na última célula

---
## Seção 1 — Detecção de Ambiente e Instalação

In [ ]:
import subprocess
import sys
import os

# ── Detecta GPU via nvidia-smi ────────────────────────────────────────────────
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=10
    )
    HAS_GPU = result.returncode == 0 and result.stdout.strip()
    GPU_INFO = result.stdout.strip() if HAS_GPU else 'N/A'
except (FileNotFoundError, subprocess.TimeoutExpired):
    HAS_GPU = False
    GPU_INFO = 'N/A'

# ── Detecta A100 para benchmark mais intenso ──────────────────────────────────
IS_A100 = HAS_GPU and 'A100' in GPU_INFO
IS_T4   = HAS_GPU and 'T4'   in GPU_INFO

print(f'╔══════════════════════════════════════════════════════╗')
print(f'║  Ambiente de Execução                               ║')
print(f'╠══════════════════════════════════════════════════════╣')
print(f'║  GPU detectada : {HAS_GPU}                                ║'.replace('True ', 'True  '))
print(f'║  GPU info      : {GPU_INFO[:40]:40s}  ║')
print(f'║  Python        : {sys.version.split()[0]:10s}                          ║')
print(f'╚══════════════════════════════════════════════════════╝')

if not HAS_GPU:
    print('\n⚠️  AVISO: GPU não detectada! Altere o runtime para GPU.')
    print('   Vá em: Ambiente de execução → Alterar tipo → GPU T4')
else:
    print(f'\n✅ GPU pronta para validação.')

In [ ]:
# ── Instalação de dependências ────────────────────────────────────────────────
# JAX com CUDA 12 se GPU disponível, jax[cpu] caso contrário
if HAS_GPU:
    print('Instalando jax[cuda12]...')
    !pip install -q 'jax[cuda12]' numba
else:
    print('Instalando jax[cpu] (sem GPU)...')
    !pip install -q 'jax[cpu]' numba

print('Instalação de dependências concluída.')

In [ ]:
# ── Clone do repositório e instalação do pacote ───────────────────────────────
import os

REPO_DIR = '/content/geosteering-ai'

if not os.path.exists(REPO_DIR):
    print('Clonando repositório...')
    # Nota: se o repositório for privado, use um token de acesso pessoal:
    # !git clone -q https://<TOKEN>@github.com/daniel-guitarplayer-8/geosteering-ai.git {REPO_DIR}
    !git clone -q https://github.com/daniel-guitarplayer-8/geosteering-ai.git {REPO_DIR}
else:
    print('Repositório já clonado. Atualizando...')
    !cd {REPO_DIR} && git pull -q origin main

os.chdir(REPO_DIR)
!pip install -q -e . 2>&1 | tail -5

# Verifica versão instalada
from geosteering_ai.simulation import __version__ as SIM_VERSION
print(f'\n✅ geosteering_ai.simulation versão: {SIM_VERSION}')
assert SIM_VERSION >= '1.5.0', f'Versão esperada ≥ 1.5.0, encontrado: {SIM_VERSION}'

---
## Seção 2 — Verificação do Ambiente JAX

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

# Habilita float64 — CRÍTICO para paridade com Numba
jax.config.update('jax_enable_x64', True)

from geosteering_ai.simulation._jax import HAS_JAX
from geosteering_ai.simulation._numba.propagation import HAS_NUMBA

print('╔══════════════════════════════════════════════════════╗')
print('║  Ambiente JAX                                       ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  JAX versão    : {jax.__version__:10s}                       ║')
print(f'║  Backend padrão: {jax.default_backend():10s}                       ║')
print(f'║  Devices       : {str(jax.devices())[:35]:35s}  ║')
print(f'║  HAS_JAX       : {str(HAS_JAX):5s}                                 ║')
print(f'║  HAS_NUMBA     : {str(HAS_NUMBA):5s}                                 ║')
print(f'║  NumPy versão  : {np.__version__:10s}                       ║')
print('╚══════════════════════════════════════════════════════╝')

BACKEND = jax.default_backend()
IS_GPU_BACKEND = BACKEND in ('gpu', 'cuda')

if IS_GPU_BACKEND:
    print(f'\n✅ JAX rodando em GPU ({BACKEND}).')
else:
    print(f'\n⚠️  JAX rodando em CPU. Gates G2 e G3 serão medidos em CPU.')
    print('   Altere o runtime para GPU para validação completa.')

assert HAS_JAX, 'JAX não está disponível!'

In [ ]:
# ── Smoke test JAX: operação simples para confirmar backend ───────────────────
x = jnp.ones((100, 100), dtype=jnp.float64)
y = jnp.dot(x, x)
y.block_until_ready()
print(f'Smoke test JAX: dot((100,100) × (100,100)) = {float(y[0,0]):.1f} ✅')
print(f'dtype float64 ativo: {y.dtype} ✅')

---
## Seção 3 — Gate G1: Consolidação XLA (44 → 1 programa)

In [ ]:
from geosteering_ai.simulation._jax import (
    clear_unified_jit_cache,
    count_compiled_xla_programs,
)
from geosteering_ai.simulation._jax.forward_pure import (
    build_static_context,
    clear_jit_cache,
    forward_pure_jax,
)
from geosteering_ai.simulation.validation.canonical_models import get_canonical_model

def _clear_all_caches():
    """Limpa caches JAX para isolar contagens XLA entre testes."""
    clear_jit_cache()
    clear_unified_jit_cache()


def _build_ctx(model_name, strategy, n_pos=100):
    """Constrói contexto estático para um modelo canônico."""
    m = get_canonical_model(model_name)
    z = np.linspace(m.min_depth - 2, m.max_depth + 2, n_pos)
    return build_static_context(
        m.rho_h, m.rho_v, m.esp, z,
        freqs_hz=np.array([20000.0]),
        tr_spacing_m=1.0, dip_deg=0.0,
        strategy=strategy,
    ), m


print('Gate G1: Verificando consolidação XLA em oklahoma_28...')
print('(oklahoma_28 tem 28 camadas → 44 buckets com strategy=bucketed)\n')

results_xla = {}
for model_name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28']:
    for strategy in ['bucketed', 'unified']:
        _clear_all_caches()
        ctx, m = _build_ctx(model_name, strategy, n_pos=100)
        # Warmup (compila o JIT)
        H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx)
        H.block_until_ready()
        xla_count = count_compiled_xla_programs(ctx)
        results_xla[(model_name, strategy)] = xla_count

print(f'{'Modelo':15s} {'Camadas':>8s} {'XLA bucketed':>14s} {'XLA unified':>13s} {'Consolidação':>14s}')
print('-' * 68)
for model_name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28']:
    m = get_canonical_model(model_name)
    xla_b = results_xla[(model_name, 'bucketed')]
    xla_u = results_xla[(model_name, 'unified')]
    consolidacao = f'{xla_b}×' if xla_u == 1 else f'{xla_b/xla_u:.1f}×'
    status = '✅' if xla_u == 1 else '❌'
    print(f'{model_name:15s} {len(m.rho_h):>8d} {xla_b:>14d} {xla_u:>13d} {consolidacao:>10s} {status}')

# Gate G1
G1_PASS = results_xla[('oklahoma_28', 'unified')] == 1
print(f'\nGate G1: XLA unified oklahoma_28 == 1: {"✅ PASS" if G1_PASS else "❌ FAIL"}')

---
## Seção 4 — Gate G4: Paridade Numérica (unified vs bucketed)

In [ ]:
print('Gate G4: Verificando paridade numérica unified vs bucketed...')
print('Gate: max|H_b - H_u| < 1e-10\n')

results_parity = {}
for model_name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28']:
    _clear_all_caches()
    ctx_b, m = _build_ctx(model_name, 'bucketed', n_pos=60)
    H_b = forward_pure_jax(ctx_b.rho_h_jnp, ctx_b.rho_v_jnp, ctx_b)
    H_b.block_until_ready()

    _clear_all_caches()
    ctx_u, _ = _build_ctx(model_name, 'unified', n_pos=60)
    H_u = forward_pure_jax(ctx_u.rho_h_jnp, ctx_u.rho_v_jnp, ctx_u)
    H_u.block_until_ready()

    max_abs = float(jnp.abs(H_b - H_u).max())
    results_parity[model_name] = max_abs

print(f'{'Modelo':15s} {'max|H_b - H_u|':>16s} {'Gate 1e-10':>12s} {'Status':>8s}')
print('-' * 56)
for model_name, max_abs in results_parity.items():
    ok = max_abs < 1e-10
    print(f'{model_name:15s} {max_abs:>16.3e} {'<1e-10':>12s} {'✅ PASS' if ok else '❌ FAIL':>8s}')

G4_PASS = all(v < 1e-10 for v in results_parity.values())
print(f'\nGate G4: Paridade <1e-10 em todos os modelos: {"✅ PASS" if G4_PASS else "❌ FAIL"}')

---
## Seção 5 — Gate G5: jacfwd Diferenciável em Alta Resistividade

In [ ]:
print('Gate G5: jacfwd unified em oklahoma_28 com ρ_high ≈ 1500 Ω·m...')
print('(Alta resistividade é cenário crítico para NaN/Inf em sensores LWD)\n')

_clear_all_caches()
m_28 = get_canonical_model('oklahoma_28')

# Escalonar resistividades para ρ > 1000 Ω·m (cenário duro)
rho_h_high = np.clip(m_28.rho_h, a_min=1.0, a_max=None) * 15.0
rho_v_high = np.clip(m_28.rho_v, a_min=1.0, a_max=None) * 15.0
z_short = np.linspace(m_28.min_depth, m_28.max_depth, 20)

ctx_jac = build_static_context(
    rho_h_high, rho_v_high, m_28.esp, z_short,
    freqs_hz=np.array([20000.0]),
    tr_spacing_m=1.0, dip_deg=0.0,
    strategy='unified',
)

def _fwd_fn(rh, rv):
    return forward_pure_jax(rh, rv, ctx_jac)

# jacfwd: diferenciação automática sobre rho_h
J = jax.jacfwd(_fwd_fn, argnums=0)(ctx_jac.rho_h_jnp, ctx_jac.rho_v_jnp)
J.block_until_ready()

n_nan = int(jnp.isnan(J).sum())
n_inf = int(jnp.isinf(J).sum())
n_total = J.size

print(f'Jacobiano shape: {J.shape}')
print(f'NaNs: {n_nan}/{n_total}')
print(f'Infs: {n_inf}/{n_total}')
print(f'Max |J|: {float(jnp.abs(J).max()):.4e}')

G5_PASS = (n_nan == 0) and (n_inf == 0)
print(f'\nGate G5: jacfwd unified alta-ρ (0 NaN, 0 Inf): {"✅ PASS" if G5_PASS else "❌ FAIL"}')

---
## Seção 6 — Gate G2: VRAM Peak (target < 1 GB para oklahoma_28)

In [ ]:
import gc

def _get_gpu_vram_mb():
    """Retorna VRAM usada em MB via nvidia-smi. Retorna None se não disponível."""
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=5
        )
        if result.returncode == 0:
            return float(result.stdout.strip().split('\n')[0])
    except Exception:
        pass
    return None


def _measure_vram_for_strategy(model_name, strategy, n_pos=100):
    """Mede VRAM peak após warm-up para um modelo + estratégia."""
    gc.collect()
    _clear_all_caches()

    vram_antes = _get_gpu_vram_mb() or 0.0

    ctx, _ = _build_ctx(model_name, strategy, n_pos=n_pos)
    H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx)
    H.block_until_ready()

    vram_depois = _get_gpu_vram_mb() or 0.0
    delta_mb = vram_depois - vram_antes

    return vram_antes, vram_depois, delta_mb


print('Gate G2: Medindo VRAM peak (oklahoma_28, 100 posições)...')
print('Target: VRAM unified < 1000 MB (≈ 250 MB esperado vs ≈ 11 GB bucketed)\n')

if not HAS_GPU:
    print('⚠️  GPU não disponível — Gate G2 não pode ser medido em CPU.')
    print('   Execute este notebook em runtime GPU T4 para validação completa.')
    G2_PASS = None  # Inconclusivo
else:
    vram_results = {}
    for strategy in ['bucketed', 'unified']:
        vram_antes, vram_depois, delta = _measure_vram_for_strategy(
            'oklahoma_28', strategy, n_pos=100
        )
        vram_results[strategy] = {
            'antes_mb': vram_antes,
            'depois_mb': vram_depois,
            'delta_mb': delta,
        }

    print(f'{'Estratégia':12s} {'VRAM antes (MB)':>17s} {'VRAM depois (MB)':>18s} {'Delta (MB)':>12s}')
    print('-' * 64)
    for s, r in vram_results.items():
        print(f'{s:12s} {r["antes_mb"]:>17.1f} {r["depois_mb"]:>18.1f} {r["delta_mb"]:>12.1f}')

    delta_u = vram_results['unified']['delta_mb']
    delta_b = vram_results['bucketed']['delta_mb']
    reducao_x = delta_b / delta_u if delta_u > 0 else float('inf')

    print(f'\nRedução VRAM: {reducao_x:.1f}× (unified vs bucketed)')

    G2_PASS = delta_u < 1000  # Gate: < 1 GB
    print(f'Gate G2: VRAM unified oklahoma_28 < 1 GB: {"✅ PASS" if G2_PASS else "❌ FAIL"}')
    print(f'  VRAM unified delta: {delta_u:.1f} MB')

---
## Seção 7 — Gate G3: Throughput GPU (target ≥ 3× bucketed)

In [ ]:
import time

def bench_strategy(model_name, strategy, n_pos=100, n_freqs=1, reps=10):
    """Mede throughput (mod/h) para um modelo + estratégia após warmup."""
    _clear_all_caches()
    m = get_canonical_model(model_name)
    z = np.linspace(m.min_depth - 2, m.max_depth + 2, n_pos)
    freqs = np.array([20000.0 * (i + 1) for i in range(n_freqs)])

    ctx = build_static_context(
        m.rho_h, m.rho_v, m.esp, z,
        freqs_hz=freqs,
        tr_spacing_m=1.0, dip_deg=0.0,
        strategy=strategy,
    )

    # Warmup (compila JIT)
    H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx)
    H.block_until_ready()

    # Medição pós-warmup
    t0 = time.perf_counter()
    for _ in range(reps):
        H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx)
    H.block_until_ready()
    elapsed = time.perf_counter() - t0

    ms_per_call = elapsed * 1e3 / reps
    modh = 3600.0 / (elapsed / reps)  # modelos/hora
    return ms_per_call, modh


print('Gate G3: Benchmark throughput unified vs bucketed...')
print(f'Dispositivo: {jax.default_backend().upper()}')
print(f'Configuração: oklahoma_3/5/28 × 100 posições × 1 freq × 10 reps\n')

bench_results = []
for model_name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28']:
    ms_b, modh_b = bench_strategy(model_name, 'bucketed', n_pos=100, reps=10)
    ms_u, modh_u = bench_strategy(model_name, 'unified',  n_pos=100, reps=10)
    ratio = modh_u / modh_b  # speedup unified vs bucketed (> 1 = unified mais rápido)
    xla_b = results_xla[(model_name, 'bucketed')]
    xla_u = results_xla[(model_name, 'unified')]
    bench_results.append({
        'model': model_name,
        'ms_b': ms_b, 'modh_b': modh_b,
        'ms_u': ms_u, 'modh_u': modh_u,
        'ratio': ratio,
        'xla_b': xla_b, 'xla_u': xla_u,
    })

print(f'{'Modelo':15s} {'XLA_b':>6s} {'XLA_u':>6s} '
      f'{'bucketed_ms':>12s} {'unified_ms':>11s} '
      f'{'speedup':>8s} {'Status':>8s}')
print('-' * 80)
for r in bench_results:
    status = '✅' if r['ratio'] >= 1.0 else ('⚠️' if r['ratio'] >= 0.5 else '❌')
    print(f'{r["model"]:15s} {r["xla_b"]:>6d} {r["xla_u"]:>6d} '
          f'{r["ms_b"]:>10.2f}ms {r["ms_u"]:>9.2f}ms '
          f'{r["ratio"]:>7.2f}× {status}')

# Gate G3: speedup oklahoma_28 >= 3× em GPU (diferente do CPU onde era 1.75×)
ratio_28 = next(r['ratio'] for r in bench_results if r['model'] == 'oklahoma_28')
G3_PASS = ratio_28 >= (3.0 if IS_GPU_BACKEND else 0.7)  # Gate CPU é mais folgado
gate_label = '≥3× (GPU)' if IS_GPU_BACKEND else '≥0.7× (CPU soft gate)'
print(f'\nGate G3: Speedup unified oklahoma_28 {gate_label}: {"✅ PASS" if G3_PASS else "❌ FAIL"}')
print(f'  Speedup medido: {ratio_28:.2f}×')

---
## Seção 8 — Benchmark Estendido (Multi-freq e Multi-posição)

In [ ]:
print('Benchmark estendido: oklahoma_28 × configurações variadas...\n')

configs_ext = [
    {'n_pos': 100,  'n_freqs': 1,  'label': 'baseline (100 pos, 1 freq)'},
    {'n_pos': 600,  'n_freqs': 1,  'label': 'grande   (600 pos, 1 freq)'},
    {'n_pos': 100,  'n_freqs': 3,  'label': 'multi-f  (100 pos, 3 freqs)'},
    {'n_pos': 600,  'n_freqs': 3,  'label': 'produção (600 pos, 3 freqs)'},
]

print(f'{'Configuração':30s} {'bucketed_ms':>12s} {'unified_ms':>12s} {'speedup':>9s}')
print('-' * 70)

for cfg in configs_ext:
    ms_b, _ = bench_strategy('oklahoma_28', 'bucketed',
                              n_pos=cfg['n_pos'], n_freqs=cfg['n_freqs'], reps=5)
    ms_u, _ = bench_strategy('oklahoma_28', 'unified',
                              n_pos=cfg['n_pos'], n_freqs=cfg['n_freqs'], reps=5)
    ratio = ms_b / ms_u  # speedup unified vs bucketed
    print(f'{cfg["label"]:30s} {ms_b:>10.2f}ms {ms_u:>10.2f}ms {ratio:>8.2f}×')

print('\n(speedup > 1 significa unified mais rápido que bucketed)')

---
## Seção 9 — Backward Compatibility (default = bucketed)

In [ ]:
print('Verificando backward compatibility: build_static_context sem strategy...')

m_3 = get_canonical_model('oklahoma_3')
z_test = np.linspace(m_3.min_depth - 2, m_3.max_depth + 2, 10)

# Sem parâmetro strategy — deve usar default "bucketed"
ctx_default = build_static_context(
    m_3.rho_h, m_3.rho_v, m_3.esp, z_test,
    freqs_hz=np.array([20000.0]),
    tr_spacing_m=1.0, dip_deg=0.0,
    # strategy omitido intencionalmente
)

print(f'ctx.strategy sem argumento: "{ctx_default.strategy}"')
COMPAT_PASS = ctx_default.strategy == 'bucketed'
print(f'Default preservado como "bucketed": {"✅ PASS" if COMPAT_PASS else "❌ FAIL"}')

# Verifica que forward roda corretamente com default
_clear_all_caches()
H_default = forward_pure_jax(ctx_default.rho_h_jnp, ctx_default.rho_v_jnp, ctx_default)
H_default.block_until_ready()
print(f'Forward com default "bucketed": ✅ OK (shape={H_default.shape})')

---
## Seção 10 — Relatório Final GO/NO-GO

In [ ]:
print('═' * 70)
print('  RELATÓRIO GO/NO-GO — JAX Unified JIT v1.5.0 → v1.5.1')
print('═' * 70)
print(f'  Dispositivo  : {jax.default_backend().upper()} ({GPU_INFO if HAS_GPU else "CPU"})')
print(f'  JAX versão   : {jax.__version__}')
print(f'  Pacote       : geosteering_ai.simulation v{SIM_VERSION}')
print('─' * 70)

# Coleta resultados
gates = {
    'G1 — XLA count == 1 (oklahoma_28 unified)': G1_PASS,
    'G4 — Paridade <1e-10 (3 modelos canônicos)': G4_PASS,
    'G5 — jacfwd sem NaN/Inf (alta-ρ unified)':   G5_PASS,
    'Backward compat — default bucketed preservado': COMPAT_PASS,
}

# Gates GPU
if G2_PASS is None:
    gates['G2 — VRAM unified < 1 GB'] = None
else:
    gates['G2 — VRAM unified < 1 GB (GPU gate)'] = G2_PASS
gates[f'G3 — Speedup unified oklahoma_28 (GPU gate)'] = G3_PASS

print()
all_pass = True
for label, result in gates.items():
    if result is None:
        status = '⚠️  N/A (sem GPU)'
    elif result:
        status = '✅ PASS'
    else:
        status = '❌ FAIL'
        all_pass = False
    print(f'  {label:<52s} {status}')

print()
print('─' * 70)

if all_pass and IS_GPU_BACKEND:
    print('  🟢  DECISÃO: GO — Todos os gates aprovados em GPU')
    print('  →  AÇÃO: Abrir PR #25 — flip jax_strategy default → "unified"')
    print('  →  PRÓXIMO: bump __version__ = "1.5.1"')
elif all_pass and not IS_GPU_BACKEND:
    print('  🟡  DECISÃO: PARCIAL — Gates CPU aprovados; pendente validação GPU')
    print('  →  AÇÃO: Re-executar este notebook em runtime GPU T4')
    print('  →  AMBIENTE: Ambiente de execução → Alterar tipo → T4 GPU')
else:
    print('  🔴  DECISÃO: NO-GO — Um ou mais gates falharam')
    print('  →  AÇÃO: Investigar os gates ❌ antes de flip de default')
    failed = [k for k, v in gates.items() if v is False]
    for f in failed:
        print(f'     • Falhou: {f}')

print('═' * 70)

In [ ]:
# ── Tabela de paridade detalhada ──────────────────────────────────────────────
print('Paridade detalhada unified vs bucketed:')
print(f'{'Modelo':15s} {'max|H_b - H_u|':>18s} {'4 ordens < gate':>17s}')
print('-' * 55)
for model_name, max_abs in results_parity.items():
    ordens = -np.log10(max_abs) - (-np.log10(1e-10))
    print(f'{model_name:15s} {max_abs:>18.3e} {ordens:>+15.1f} ordens')

print()
print('XLA consolidação:')
print(f'{'Modelo':15s} {'bucketed':>10s} {'unified':>10s} {'consolidação':>14s}')
print('-' * 55)
for model_name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28']:
    xla_b = results_xla[(model_name, 'bucketed')]
    xla_u = results_xla[(model_name, 'unified')]
    print(f'{model_name:15s} {xla_b:>10d} {xla_u:>10d} {xla_b}× → 1')

In [ ]:
# ── Limpeza final de caches ───────────────────────────────────────────────────
_clear_all_caches()
print('Caches JAX limpos.')
print('Notebook de validação concluído. ✅')